# Module 18 - Tool use

Use this notebook after `tests/test_tools.py` is passing. The notebook starts with deterministic local checks for schemas, parsers, dispatch, and the tool loop, then moves into optional ProdLM runs where a real local model has to choose when to call tools.

The deliverable is a tool-use postmortem: what tools you exposed, where the model used them correctly, where it failed, and what you would improve before turning this into the Module 19 agent loop.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import subprocess
import sys

from IPython.display import Markdown, display

from g2c.inference import Backend, BackendInfo, InferenceResult, load_prodlm_backend, prodlm_manifest_exists
from g2c.notebook_extras.sampling import printable
from g2c.tools import (
    DEFAULT_SYSTEM,
    Tool,
    ToolCall,
    ToolRegistry,
    calculator_evaluate,
    dispatch_tool_call,
    format_tool_results,
    make_calculator,
    make_read_file,
    make_run_python,
    make_web_search,
    parse_tool_calls,
    render_tools_for_prompt,
    run_with_tools,
    validate_arguments,
)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

/Users/colkitt/sith/toys/courses/g2c


Run the tool tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 18 TODOs in `g2c/tools/`.

In [2]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_tools.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 18 tool tests are not passing yet."


........................................................................ [ 46%]
........................................................................ [ 92%]
...........                                                              [100%]



## Display helpers

In [3]:
def short(text: Any, limit: int = 180) -> str:
    rendered = printable(str(text)).replace("\n", "\\n")
    if len(rendered) <= limit:
        return rendered
    return rendered[: limit - 3] + "..."


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    def cell(value: Any) -> str:
        text = str(value).replace("|", "\\|").replace("\n", "<br>")
        return text

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(cell(row.get(col, "")) for col in columns) + " |" for row in rows]
    return "\n".join([header, sep, *body])


def show_tool(tool: Tool) -> None:
    props = tool.parameters.get("properties", {})
    required = set(tool.parameters.get("required", []))
    rows = []
    for name, spec in props.items():
        rows.append(
            {
                "argument": name,
                "type": spec.get("type", ""),
                "required": "yes" if name in required else "no",
                "description": spec.get("description", ""),
            }
        )
    display(Markdown(f"### `{tool.name}`\n\n{tool.description}"))
    if rows:
        display(Markdown(markdown_table(rows, ["argument", "type", "required", "description"])))
    else:
        display(Markdown("No arguments."))


def show_registry(registry: ToolRegistry) -> None:
    display(Markdown(f"Registered tools: `{', '.join(registry.names())}`"))
    for tool in registry:
        show_tool(tool)


def show_tool_run(result) -> None:
    print("stopped:", result.stopped_reason)
    print("final answer:", printable(result.final_answer or "(none)"))
    print()
    rows = []
    for step_index, step in enumerate(result.steps, start=1):
        if not step.tool_calls:
            rows.append(
                {
                    "step": step_index,
                    "tool": "(none)",
                    "arguments": "",
                    "error": "",
                    "output": short(step.completion),
                }
            )
            continue
        for call, tool_result in zip(step.tool_calls, step.tool_results):
            rows.append(
                {
                    "step": step_index,
                    "tool": call.name,
                    "arguments": json.dumps(call.arguments),
                    "error": "yes" if tool_result.is_error else "no",
                    "output": short(tool_result.output),
                }
            )
    display(Markdown(markdown_table(rows, ["step", "tool", "arguments", "error", "output"])))


## Build a local tool registry

Start with the three tools that let an assistant leave pure text generation: a calculator, a sandboxed file reader, and a small Python runner. The registry is both the prompt source and the dispatch table.

In [4]:
scratch_dir = repo_root / "data" / "module18-tools"
scratch_dir.mkdir(parents=True, exist_ok=True)
(scratch_dir / "numbers.txt").write_text("12\n19\n31\n44\n", encoding="utf-8")
(scratch_dir / "sales.csv").write_text(
    "item,units,price\nnotebook,3,7.50\npen,12,1.25\nmarker,5,2.00\n",
    encoding="utf-8",
)

registry = ToolRegistry(
    [
        make_calculator(),
        make_read_file(root=scratch_dir),
        make_run_python(timeout=5.0),
        make_web_search(),
    ]
)
show_registry(registry)


Registered tools: `calculator, read_file, run_python, web_search`

### `calculator`

Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.

| argument | type | required | description |
| --- | --- | --- | --- |
| expression | string | yes | Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'. |

### `read_file`

Read a UTF-8 text file from the project root and return its contents. The path is interpreted relative to a sandboxed root directory.

| argument | type | required | description |
| --- | --- | --- | --- |
| path | string | yes | Relative path under the project root. |
| max_chars | integer | no | Optional truncation length (default 10000). |

### `run_python`

Execute a Python snippet in a subprocess and return stdout. Use print() to emit results. Has a wall-clock timeout.

| argument | type | required | description |
| --- | --- | --- | --- |
| code | string | yes | Python source to run. |

### `web_search`

Search the web for the given query. (Stub by default — wire a real backend for real results.)

| argument | type | required | description |
| --- | --- | --- | --- |
| query | string | yes | Search query. |

The model sees a rendered version of that registry inside the system prompt. This is the contract it must follow to call tools.

In [5]:
print(render_tools_for_prompt(registry.tools))

Tools available:
- calculator: Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.
  parameters:
  {
    "type": "object",
    "properties": {
      "expression": {
        "type": "string",
        "description": "Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'."
      }
    },
    "required": [
      "expression"
    ]
  }
- read_file: Read a UTF-8 text file from the project root and return its contents. The path is interpreted relative to a sandboxed root directory.
  parameters:
  {
    "type": "object",
    "properties": {
      "path": {
        "type": "string",
        "description": "Relative path under the project root."
      },
      "max_chars": {
        "type": "integer",
        "description": "Optional truncation length (default 10000)."
      }
    },
    "required": [
      "path"
    ]
  }
- run_python: Execute a Python snippet in a subprocess and return stdout. Use print() to emit re

## Exercise 1 - Validate arguments

The parser should not decide whether arguments are safe for a specific tool. Validation needs the tool schema, so it happens after parsing and before dispatch.

In [6]:
calculator = registry.get("calculator")

valid_args = {"expression": "(23 * 17) + 5"}
validated = validate_arguments(calculator, valid_args)
print("validated:", validated)

bad_argument_sets = [
    {},
    {"expr": "23 * 17"},
    {"expression": 391},
    {"expression": "23 * 17", "round": True},
]

for args in bad_argument_sets:
    try:
        validate_arguments(calculator, args)
    except Exception as exc:
        print(f"{args!r} -> {type(exc).__name__}: {exc}")

validated: {'expression': '(23 * 17) + 5'}
{} -> ToolError: missing required arguments: ['expression']
{'expr': '23 * 17'} -> ToolError: missing required arguments: ['expression']
{'expression': 391} -> ToolError: argument 'expression': expected string, got int
{'expression': '23 * 17', 'round': True} -> ToolError: unknown arguments: ['round']


## Exercise 2 - Parse tool calls

A model output is just text. The parser extracts every well-formed `<tool_call>...</tool_call>` block, parses the JSON body, checks the shape, and assigns a `call_id`.

In [7]:
model_text = (
    "I should use the calculator.\n"
    "<tool_call>\n"
    '{"name": "calculator", "arguments": {"expression": "23 * 17"}}\n'
    "</tool_call>\n"
)

calls = parse_tool_calls(model_text)
for call in calls:
    print(call)

malformed_examples = [
    "<tool_call>not json</tool_call>",
    '<tool_call>{"arguments": {}}</tool_call>',
    '<tool_call>{"name": "calculator", "arguments": []}</tool_call>',
]

for text in malformed_examples:
    print(short(text), "->", parse_tool_calls(text))


ToolCall(name='calculator', arguments={'expression': '23 * 17'}, call_id='call_0_f2cf6add')
<tool_call>not json</tool_call> -> []
<tool_call>{"arguments": {}}</tool_call> -> []
<tool_call>{"name": "calculator", "arguments": []}</tool_call> -> []


## Exercise 3 - Dispatch calls and surface errors as data

`dispatch_tool_call` turns every outcome into a `ToolResult`. Unknown tools, malformed arguments, and tool exceptions become `is_error=True` results instead of crashing the loop.

In [8]:
manual_calls = [
    ToolCall(name="calculator", arguments={"expression": "23 * 17"}, call_id="manual_ok"),
    ToolCall(name="calculator", arguments={"expr": "23 * 17"}, call_id="manual_bad_args"),
    ToolCall(name="unknown_tool", arguments={}, call_id="manual_unknown"),
]

results = [dispatch_tool_call(registry, call) for call in manual_calls]
for result in results:
    print(result)

print()
print(format_tool_results(results))

ToolResult(call_id='manual_ok', name='calculator', output='391', is_error=False)
ToolResult(call_id='manual_bad_args', name='calculator', output="missing required arguments: ['expression']", is_error=True)
ToolResult(call_id='manual_unknown', name='unknown_tool', output='"no tool named \'unknown_tool\'; registered: [\'calculator\', \'read_file\', \'run_python\', \'web_search\']"', is_error=True)

<tool_result name="calculator" id="manual_ok">
391
</tool_result>

<tool_error name="calculator" id="manual_bad_args">
missing required arguments: ['expression']
</tool_error>

<tool_error name="unknown_tool" id="manual_unknown">
"no tool named 'unknown_tool'; registered: ['calculator', 'read_file', 'run_python', 'web_search']"
</tool_error>


## Exercise 4 - Safe calculator behavior

The calculator is deliberately not `eval`. It parses Python expression syntax into an AST and only accepts a small arithmetic whitelist.

In [9]:
safe_expressions = [
    "1 + 2",
    "(2 + 3) * 4",
    "2 ** 10",
    "7 / 2",
    "7 // 2",
    "-5 + 3",
]

unsafe_expressions = [
    "os.system('ls')",
    "__import__('os').system('ls')",
    "open('/etc/passwd').read()",
    "True + 1",
    "'hello'",
]

for expression in safe_expressions:
    print(expression, "=>", calculator_evaluate(expression))

print()
for expression in unsafe_expressions:
    try:
        calculator_evaluate(expression)
    except Exception as exc:
        print(expression, "=>", type(exc).__name__, exc)

1 + 2 => 3
(2 + 3) * 4 => 20
2 ** 10 => 1024
7 / 2 => 3.5
7 // 2 => 3
-5 + 3 => -2

os.system('ls') => ToolError AST node not allowed: Call
__import__('os').system('ls') => ToolError AST node not allowed: Call
open('/etc/passwd').read() => ToolError AST node not allowed: Call
True + 1 => ToolError booleans are not allowed in expressions
'hello' => ToolError constant must be a number, got str


## Exercise 5 - Run the loop with a fake backend

Before involving a real model, use a deterministic backend that emits known completions. This isolates the loop contract: complete, parse, dispatch, feed back, repeat.

In [10]:
class FakeBackend(Backend):
    def __init__(self, completions: list[str], *, model_id: str = "fake-tools") -> None:
        self._completions = list(completions)
        self._info = BackendInfo(name="fake", model_id=model_id)
        self.calls: list[dict[str, Any]] = []

    @property
    def info(self) -> BackendInfo:
        return self._info

    def complete(
        self,
        prompt: str,
        *,
        max_new_tokens: int = 128,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
    ) -> InferenceResult:
        if not self._completions:
            raise RuntimeError("FakeBackend has no completions left")
        completion = self._completions.pop(0)
        self.calls.append(
            {
                "prompt": prompt,
                "max_new_tokens": max_new_tokens,
                "temperature": temperature,
                "top_k": top_k,
                "top_p": top_p,
            }
        )
        return InferenceResult(
            prompt=prompt,
            completion=completion,
            prompt_tokens=len(prompt.split()),
            completion_tokens=len(completion.split()),
            latency_ms=1.0,
            backend=self._info,
        )

In [11]:
fake_backend = FakeBackend(
    [
        '<tool_call>{"name": "calculator", "arguments": {"expression": "23 * 17"}}</tool_call>',
        "23 * 17 = 391.",
    ]
)

fake_result = run_with_tools(
    fake_backend,
    registry,
    "What is 23 times 17?",
    max_steps=3,
    temperature=0.0,
)
show_tool_run(fake_result)

print("\nPrompt after tool feedback included:")
print(short(fake_backend.calls[-1]["prompt"], limit=900))


stopped: no_more_calls
final answer: 23 * 17 = 391.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | calculator | {"expression": "23 * 17"} | no | 391 |
| 2 | (none) |  |  | 23 * 17 = 391. |


Prompt after tool feedback included:
You are a helpful assistant with access to tools. To call a tool, emit a <tool_call> block whose body is JSON of the form {"name": "<tool>", "arguments": {...}}. The arguments must match the tool's parameters schema exactly. After you emit the block, stop and wait for the result. When you have enough information to answer the user, give your final answer in plain text without any <tool_call> blocks.\n\nTools available:\n- calculator: Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.\n  parameters:\n  {\n    "type": "object",\n    "properties": {\n      "expression": {\n        "type": "string",\n        "description": "Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'."\n      }\n    },\n    "required": [\n      "expression"\n    ]\n  }\n- read_file: Read a UTF-8 text file from the project root and retu...


The fake backend can also model recovery from an error. The first call uses the wrong argument name, the error is fed back, and the next turn corrects it.

In [12]:
recovering_backend = FakeBackend(
    [
        '<tool_call>{"name": "calculator", "arguments": {"expr": "23 * 17"}}</tool_call>',
        '<tool_call>{"name": "calculator", "arguments": {"expression": "23 * 17"}}</tool_call>',
        "The answer is 391.",
    ],
    model_id="fake-recovery",
)

recovery_result = run_with_tools(
    recovering_backend,
    registry,
    "What is 23 times 17?",
    max_steps=4,
    temperature=0.0,
)
show_tool_run(recovery_result)

stopped: no_more_calls
final answer: The answer is 391.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | calculator | {"expr": "23 * 17"} | yes | missing required arguments: ['expression'] |
| 2 | calculator | {"expression": "23 * 17"} | no | 391 |
| 3 | (none) |  |  | The answer is 391. |

## Exercise 6 - Load ProdLM for live tool use

The fake backend proves your loop. ProdLM tests whether a real instruction model will choose the right tool format. Run `./prodlm.sh` first if this cell says ProdLM is not configured.

In [13]:
RUN_PRODLM_EXAMPLES = prodlm_manifest_exists(repo_root=repo_root)
PRODLM_MODEL_ID = None  # set to an Ollama tag to override the configured ProdLM model

prodlm_backend = None
if RUN_PRODLM_EXAMPLES:
    prodlm_backend = load_prodlm_backend(
        repo_root=repo_root,
        model_id=PRODLM_MODEL_ID,
        required=False,
    )
    print("loaded:", prodlm_backend.info)
else:
    print("ProdLM examples disabled or not configured. Run ./prodlm.sh, then set RUN_PRODLM_EXAMPLES = True.")

loaded: BackendInfo(name='prodlm', model_id='llama3.2:3b', extra={'base_url': 'http://localhost:11434', 'configured_name': 'ProdLM'})


In [14]:
STRICT_TOOL_SYSTEM = (
    DEFAULT_SYSTEM
    + "\nFor arithmetic, exact file contents, and Python data tasks, call the available tool instead of guessing. "
    + "Emit only one tool call, then wait for the result."
)


def run_live_tool_question(question: str, *, tools: ToolRegistry = registry, max_steps: int = 5):
    if prodlm_backend is None:
        print("No ProdLM backend loaded.")
        return None
    try:
        result = run_with_tools(
            prodlm_backend,
            tools,
            question,
            system=STRICT_TOOL_SYSTEM,
            max_steps=max_steps,
            max_new_tokens=256,
            temperature=0.0,
        )
    except Exception as exc:
        print(f"ProdLM tool run failed: {type(exc).__name__}: {exc}")
        return None
    show_tool_run(result)
    return result


Start with arithmetic. You are looking for three things: whether the model calls `calculator`, whether the JSON validates, and whether it stops after receiving the result.

In [15]:
arithmetic_question = "What is (1847 * 29) - 138? Use the calculator."
arithmetic_result = run_live_tool_question(arithmetic_question, max_steps=3)

stopped: no_more_calls
final answer: The result of (1847 * 29) - 138 is 53425.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | calculator | {"expression": "(1847 * 29) - 138"} | no | 53425 |
| 2 | (none) |  |  | The result of (1847 * 29) - 138 is 53425. |

## Exercise 7 - Read, compute, answer

Now require two capabilities: reading a local file and doing arithmetic over the contents. Depending on the model, it may use `read_file` first and then either `calculator` or `run_python`.

In [16]:
file_question = "The file numbers.txt contains one number per line. Read it and report the sum and mean."
file_result = run_live_tool_question(file_question, max_steps=5)

stopped: no_more_calls
final answer: The sum of the numbers in the file is 106. The mean is 26.5.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 1 | calculator | {"expression": "(sum([int(line) for line in open('numbers.txt').read().split()]))/len(open('numbers.txt').read().split())"} | yes | AST node not allowed: Call |
| 1 | run_python | {"code": "import statistics; print(statistics.mean([int(line) for line in open('numbers.txt').read().split()]))"} | no | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 1, in <module>\nFileNotFoundError: [Errno 2] No such file or directory: 'numbers.txt' |
| 2 | (none) |  |  | The sum of the numbers in the file is 106. The mean is 26.5. |

## Exercise 8 - Python as a data tool

`run_python` is powerful, but it is not a real sandbox. In this local course setting it is useful for small data tasks; in production it would need much stronger isolation.

In [17]:
python_question = "Read sales.csv, compute total revenue as units times price for each row, and report the total."
python_result = run_live_tool_question(python_question, max_steps=5)

stopped: max_steps
final answer: (none)



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | read_file | {"path": "sales.csv"} | no | item,units,price\nnotebook,3,7.50\npen,12,1.25\nmarker,5,2.00\n |
| 1 | run_python | {"code": "import pandas as pd; df = pd.read_csv('sales.csv'); total_revenue = (df['units'] * df['price']).sum(); print(total_revenue)"} | no | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/Users/colkitt/sith/toys/courses/g2c/.venv/lib/python3.11/site-packages... |
| 2 | read_file | {"path": "sales.csv"} | no | item,units,price\nnotebook,3,7.50\npen,12,1.25\nmarker,5,2.00\n |
| 2 | run_python | {"code": "import pandas as pd; df = pd.read_csv('sales.csv'); total_revenue = (df['units'] * df['price']).sum(); print(total_revenue)"} | no | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/Users/colkitt/sith/toys/courses/g2c/.venv/lib/python3.11/site-packages... |
| 3 | read_file | {"path": "sales.csv"} | no | item,units,price\nnotebook,3,7.50\npen,12,1.25\nmarker,5,2.00\n |
| 4 | read_file | {"path": "sales.csv"} | no | item,units,price\nnotebook,3,7.50\npen,12,1.25\nmarker,5,2.00\n |
| 4 | run_python | {"code": "import pandas as pd; df = pd.read_csv('sales.csv'); total_revenue = (df['units'] * df['price']).sum(); print(total_revenue)"} | no | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/Users/colkitt/sith/toys/courses/g2c/.venv/lib/python3.11/site-packages... |
| 5 | read_file | {"path": "sales.csv"} | no | item,units,price\nnotebook,3,7.50\npen,12,1.25\nmarker,5,2.00\n |
| 5 | run_python | {"code": "import pandas as pd; df = pd.read_csv('sales.csv'); total_revenue = (df['units'] * df['price']).sum(); print(total_revenue)"} | no | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/Users/colkitt/sith/toys/courses/g2c/.venv/lib/python3.11/site-packages... |

## Exercise 9 - Add a custom tool

A tool is just a callable plus a schema. This small custom tool reverses text; replace it with a tool that would be useful in your own workflow.

In [ ]:
def make_reverse_text() -> Tool:
    def _func(text: str) -> str:
        return text[::-1]

    return Tool(
        name="reverse_text",
        description="Reverse the characters in a string.",
        parameters={
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "Text to reverse."},
            },
            "required": ["text"],
        },
        func=_func,
    )

custom_registry = ToolRegistry([make_reverse_text(), make_calculator()])
show_registry(custom_registry)

custom_call = ToolCall(
    name="reverse_text",
    arguments={"text": "tool use"},
    call_id="custom_0",
)
print(dispatch_tool_call(custom_registry, custom_call))

In [ ]:
custom_question = "Reverse the text 'agent loop', then tell me what 14 * 19 is."
custom_result = run_live_tool_question(custom_question, tools=custom_registry, max_steps=5)

## Exercise 10 - Toolformer-style ablation

Run the same question with and without tools. The important comparison is not whether ProdLM can do the task unaided once. It is whether the tool path is more reliable on exact arithmetic, file contents, and data transformation.

In [ ]:
ablation_question = "What is 8437 * 29? Give only the final integer."

if prodlm_backend is None:
    print("No ProdLM backend loaded.")
else:
    try:
        direct = prodlm_backend.complete(
            ablation_question,
            max_new_tokens=80,
            temperature=0.0,
        )
        print("direct completion:")
        print(printable(direct.completion))
        print("\nwith tools:")
        ablation_result = run_live_tool_question(ablation_question, tools=ToolRegistry([make_calculator()]), max_steps=3)
    except Exception as exc:
        print(f"ablation failed: {type(exc).__name__}: {exc}")


## Exercise 11 - Short evaluation session

Use this cell to collect a small transcript for your postmortem. Keep the question list mixed: some should require tools, and some should not.

Requires a ProdLM backend; the cell prints a friendly skip message if none is loaded.

In [19]:
session_questions = [
    "What is 511 * 37?",
    "Read numbers.txt and report the largest number.",
    "Read sales.csv and compute total revenue.",
    "What is the capital of France? Do not use tools unless needed.",
    "Try to read missing.txt and then explain what happened.",
]

transcript = []
if prodlm_backend is None:
    print("No ProdLM backend loaded.")
else:
    for question in session_questions:
        print("=" * 80)
        print(question)
        result = run_live_tool_question(question, max_steps=5)
        transcript.append(result)

What is 511 * 37?
stopped: no_more_calls
final answer: <tool_call name="calculator" arguments={"expression": "511 * 37"}></tool_call>



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | (none) |  |  | <tool_call name="calculator" arguments={"expression": "511 * 37"}></tool_call> |

Read numbers.txt and report the largest number.
stopped: max_steps
final answer: (none)



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 2 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 3 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 3 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 4 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 4 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 4 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 4 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 5 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 5 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |
| 5 | read_file | {"path": "numbers.txt"} | no | 12\n19\n31\n44\n |

Read sales.csv and compute total revenue.
stopped: no_more_calls
final answer: The total revenue is $22.50.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | read_file | {"path": "sales.csv"} | no | item,units,price\nnotebook,3,7.50\npen,12,1.25\nmarker,5,2.00\n |
| 1 | calculator | {"expression": "(total * price)"} | yes | AST node not allowed: Name |
| 1 | run_python | {"code": "import pandas as pd; df = pd.read_csv('sales.csv'); total_revenue = (df['total'] * df['price']).sum(); print(total_revenue)"} | no | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/Users/colkitt/sith/toys/courses/g2c/.venv/lib/python3.11/site-packages... |
| 2 | (none) |  |  | The total revenue is $22.50. |

What is the capital of France? Do not use tools unless needed.
stopped: no_more_calls
final answer: The capital of France is Paris.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | web_search | {"query": "capital of France"} | no | [stub web_search result for 'capital of France']\nThis is a stub. Wire DuckDuckGo / SerpAPI / Tavily / Bing into make_web_search(search=...) to get real results. |
| 1 | read_file | {"path": "data/countries.json", "max_chars": 10000} | yes | file not found: 'data/countries.json' |
| 1 | calculator | {"expression": "Paris"} | yes | AST node not allowed: Name |
| 2 | (none) |  |  | The capital of France is Paris. |

Try to read missing.txt and then explain what happened.
stopped: no_more_calls
final answer: The tool call to read missing.txt failed because the file was not found. The calculator tool call resulted in an error because it expects a numeric constant, but received None instead. This suggests that the Python code being executed by the calculator tool did not produce a valid number.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | read_file | {"path": "missing.txt"} | yes | file not found: 'missing.txt' |
| 1 | calculator | {"expression": "None"} | yes | constant must be a number, got NoneType |
| 2 | (none) |  |  | The tool call to read missing.txt failed because the file was not found. The calculator tool call resulted in an error because it expects a numeric constant, but received None i... |

## Postmortem notes

Write `docs/tools-postmortem.md` in 3-4 paragraphs. Cover:

- What you wired up: model, tools, max step budget, and system prompt.
- What worked: tasks where the model reliably called the right tool.
- Where it broke: malformed JSON, wrong tool, missing tool, looping, or stopping too early.
- What you would improve before Module 19: ReAct formatting, better stop criteria, stricter parsing, more tool-specific examples, or richer evals.